<h1>Chapter 9 - Graph RAG: Recipe 2: Enrich Company Data</h1>
<i>Building knowledge graphs and using them in RAG systems.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch09_graph_rag/9.2_enrich_company_data.ipynb)

---

This notebook is for Chapter 9 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


# Recipe 2: Enrich Company Data in Knowledge Graph

This notebook demonstrates how to build an enriched knowledge graph in Neo4j by loading company information, addresses, spend data, and SLA metadata. You'll learn how to create nodes for different entities and establish relationships between them to create a comprehensive business data graph.

## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [1]:
%pip install pandas python-dotenv neo4j

  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached neo4j-6.0.3-py3-none-any.whl.metadata (5.2 kB)
  Using cached numpy-2.3.5-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl (11.0 MB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Using cached neo4j-6.0.3-py3-none-any.whl (325 kB)
Using cached numpy-2.3.5-cp313-cp313-win_amd64.whl (12.8 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/6 [pytz]
   ---------------------------------------- 0/6 [pytz]
   ---------------------------------------- 0/6 [pytz]
   ------ --------------------------------- 1/6 [tzdata]
   ------ ------------------

## 1. Setup and Imports

Import required libraries and load environment variables.

In [2]:
from __future__ import annotations
import os
from typing import Any, Dict, Optional
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load environment variables from .env file
load_dotenv()

True

## 2. Connect to Neo4j

Create a reusable driver function that reads connection details from environment variables.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
```

In [3]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    pwd = os.getenv("NEO4J_PASSWORD")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Initialize driver
driver = get_driver()
print("✓ Connected to Neo4j")

✓ Connected to Neo4j


## 3. Load Companies

Import company information and create `Company` nodes with attributes such as name, country, and industry.

In [7]:
def load_companies(driver, companies_csv_path: str) -> None:
    """Load companies from CSV and create Company nodes."""
    df = pd.read_csv(companies_csv_path)
    print(f"Loading {len(df)} companies...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MERGE (c:Company {supplier_id: row.supplier_id})
            ON CREATE SET c.name=row.name, c.country=row.country, c.industry=row.industry
            SET c.industry = row.industry
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Companies loaded")

# Execute
companies_csv_path = "sample_data/companies.csv"
load_companies(driver, companies_csv_path)

Loading 4 companies...
✓ Companies loaded
✓ Companies loaded


## 4. Load Addresses

Create `Address` nodes with street, city, postal code, and country information.

In [8]:
def load_addresses(driver, addresses_csv_path: str) -> None:
    """Load addresses from CSV and create Address nodes."""
    df = pd.read_csv(addresses_csv_path)
    print(f"Loading {len(df)} addresses...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MERGE (a:Address {id: row.address_id})
            ON CREATE SET a.street = row.street, 
                          a.city = row.city, 
                          a.postal_code = row.postal_code, 
                          a.country = row.country
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Addresses loaded")

# Execute
addresses_csv_path = "sample_data/addresses.csv"
load_addresses(driver, addresses_csv_path)

Loading 4 addresses...
✓ Addresses loaded


## 5. Connect Companies and Addresses

Link companies to their addresses with `LOCATED_AT` relationships.

In [9]:
def connect_company_addresses(driver, companies_csv_path: str) -> None:
    """Create LOCATED_AT relationships between companies and addresses."""
    df = pd.read_csv(companies_csv_path)[["supplier_id", "address_id"]]
    print(f"Connecting {len(df)} company-address relationships...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MATCH (c:Company {supplier_id: row.supplier_id})
            MATCH (a:Address {id: row.address_id})
            MERGE (c)-[:LOCATED_AT]->(a)
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Company-address relationships created")

# Execute
companies_csv_path = "sample_data/companies.csv"
connect_company_addresses(driver, companies_csv_path)

Connecting 4 company-address relationships...
✓ Company-address relationships created
✓ Company-address relationships created


## 6. Add Spend Information

Attach yearly spend data to `Company` nodes as properties.

In [10]:
def load_spend(driver, spend_csv_path: str) -> None:
    """Load spend data and update Company nodes with spend information."""
    df = pd.read_csv(spend_csv_path)
    print(f"Loading spend data for {len(df)} companies...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MATCH (c:Company {supplier_id: row.supplier_id})
            SET c.spend_2024 = toFloat(row.spend_eur), 
                c.spend_category = row.spend_category
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Spend data loaded")

# Execute
spend_csv_path = "sample_data/spend_2024.csv"
load_spend(driver, spend_csv_path)

Loading spend data for 4 companies...
✓ Spend data loaded


## 7. Link Suppliers to SLAs

Import SLA metadata and create `HAS_SLA` relationships connecting suppliers to their contracts.

In [11]:
def load_slas(driver, sla_csv_path: str) -> None:
    """Load SLA metadata and link to Company nodes."""
    df = pd.read_csv(sla_csv_path)
    df["effective_date"] = df["effective_date"].replace({"N/A": None})
    df["effective_date"] = df["effective_date"].where(
        df["effective_date"].notna(), None
    )
    print(f"Loading {len(df)} SLA records...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MATCH (c:Company {supplier_id: row.supplier_id})
            MERGE (s:SLA {id: row.sla_id})
            ON CREATE SET s.title = row.title, 
                          s.service_name = row.service_name,
                          s.governing_law = row.governing_law,
                          s.effective_date = CASE 
                            WHEN row.effective_date IS NOT NULL 
                            THEN date(row.effective_date) 
                            ELSE NULL END
            MERGE (c)-[:HAS_SLA]->(s)
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ SLA data loaded and linked to companies")

# Execute
sla_csv_path = "sample_data/slas.csv"
load_slas(driver, sla_csv_path)

Loading 4 SLA records...
✓ SLA data loaded and linked to companies


## 8. Verify the Graph Structure

Run a query to verify that the data was loaded correctly.

In [12]:
def verify_graph_structure(driver):
    """Query the graph to verify the structure."""
    with driver.session() as session:
        # Count nodes
        result = session.run("""
            MATCH (c:Company) 
            RETURN count(c) AS company_count
        """)
        company_count = result.single()["company_count"]
        
        result = session.run("""
            MATCH (a:Address) 
            RETURN count(a) AS address_count
        """)
        address_count = result.single()["address_count"]
        
        result = session.run("""
            MATCH (s:SLA) 
            RETURN count(s) AS sla_count
        """)
        sla_count = result.single()["sla_count"]
        
        # Sample query
        result = session.run("""
            MATCH (c:Company)-[:LOCATED_AT]->(a:Address)
            MATCH (c)-[:HAS_SLA]->(s:SLA)
            RETURN c.name AS company, 
                   a.country AS country, 
                   c.spend_2024 AS spend,
                   s.title AS sla_title
            LIMIT 5
        """)
        samples = [record.data() for record in result]
        
        print(f"\n📊 Graph Statistics:")
        print(f"  Companies: {company_count}")
        print(f"  Addresses: {address_count}")
        print(f"  SLAs: {sla_count}")
        print(f"\n📋 Sample Records:")
        for i, sample in enumerate(samples, 1):
            print(f"  {i}. {sample['company']} ({sample['country']}) - {sample['sla_title']}")
            print(f"     Spend: €{sample['spend']:,.2f}" if sample['spend'] else "     Spend: N/A")

# Execute verification
verify_graph_structure(driver)


📊 Graph Statistics:
  Companies: 4
  Addresses: 4
  SLAs: 4

📋 Sample Records:
  1. Alpha Cloud GmbH (Germany) - Cloud Compute Service Level Agreement (SLA)
     Spend: €1,200,000.00
  2. DataStore BV (Netherlands) - Cloud Storage SLA
     Spend: €800,000.00
  3. ComputeNow AG (Switzerland) - VM Hosting SLA
     Spend: €450,000.00
  4. EuroCompute S.A. (France) - Legacy Compute SLA
     Spend: €300,000.00


## 9. Cleanup

Close the Neo4j driver connection.

In [13]:
driver.close()
print("✓ Connection closed")

✓ Connection closed
